# Causal Inference II: Causal Graphs and Estimating Effects


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/15-Causal-Inference-II.ipynb)

In [Causal Inference I](14-Causal-Inference-I.qmd) we saw that the naive difference in
means equals the true effect *plus selection bias*, and that **randomization** removes that
bias. But we often cannot run an experiment:

* it would be **unethical** (randomly assign people to smoke),
* it is **impossible** (we can't re-run a minimum-wage law),
* it is **too expensive or slow**, or
* the treatment **already happened** and we only have observational data.


This lecture is about doing causal inference *without* an experiment — which forces us to be
explicit about our assumptions using **causal graphs**.



## Drawing our assumptions: causal graphs

A **causal graph** is a *directed acyclic graph* (DAG): nodes are variables, and an arrow
$X \rightarrow Y$ means "$X$ is a direct cause of $Y$" [@pearl2016primer].

The graph encodes **assumptions** we bring to the data (from domain knowledge). The data
alone cannot draw it for us — but once drawn, the graph tells us *exactly* which variables
to control for.


We will study graphs in their own right later in the course. For now you need only the
handful of terms on the next slide.


## Reading a causal graph

In [ ]:
#| echo: false
#| fig-align: center
#| label: example-dag
import networkx as nx
import matplotlib.pyplot as plt

def draw(ax, edges, pos, title="", node_color="#cde", node_size=1400, font_size=12):
    """Lecture-wide DAG style: pale nodes, dark outlines, black arrows."""
    G = nx.DiGraph(edges)
    colors = [node_color.get(v, "#cde") for v in G] if isinstance(node_color, dict) else node_color
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=node_size, font_size=font_size, arrowsize=22,
            edgecolors="#333", linewidths=1.2)
    ax.set_title(title, fontsize=11)

example_edges = [("A\nage", "X\nexercise"), ("A\nage", "Y\nheart"),
                 ("X\nexercise", "W\nweight"), ("W\nweight", "Y\nheart"),
                 ("M\nmotivation", "X\nexercise"), ("M\nmotivation", "D\ndiet"),
                 ("D\ndiet", "Y\nheart")]
example_pos = {"A\nage": (1, -1.3), "M\nmotivation": (-0.6, 1.4), "D\ndiet": (0.7, 1.0),
               "X\nexercise": (0, 0), "W\nweight": (1, 0), "Y\nheart": (2, 0)}

def draw_example():
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    draw(ax, example_edges, example_pos, node_size=3000, font_size=9)
    ax.set_xlim(-1.05, 2.4); ax.set_ylim(-1.75, 1.8)
    plt.tight_layout(); plt.show()

draw_example()

* **Node** — a variable. **Arrow (directed edge)** $A \to X$ — "$A$ is a direct cause of $X$."
  $A$ is a **parent** of $X$; $X$ is a **child** of $A$.
* **Directed path** — follow the arrows head-to-tail: $X \to W \to Y$. Everything reachable
  this way from $X$ is a **descendant** of $X$ (here $W$ and $Y$); $X$'s **ancestors** are
  $A$ and $M$.


## Paths, and what makes it a DAG

In [ ]:
#| echo: false
#| fig-align: center
draw_example()

* **Acyclic** — no directed path returns to where it started: nothing is its own ancestor.
  (Cause precedes effect.) That is what makes it a **DAG**.
* **Path** (undirected) — a chain of arrows connecting two nodes, ignoring direction and
  visiting no node twice: $X \leftarrow A \to Y$ and $X \leftarrow M \to D \to Y$ are paths
  from $X$ to $Y$ too.


Causal effects flow along **directed** paths. Spurious associations can flow along the
*other* paths — but only when those paths are **open**, which is the next idea.
Remarkably, **every path is built from just three elementary structures.**


## The three building blocks

In [ ]:
#| echo: false
#| fig-align: center
#| label: building-blocks
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))

# Chain (mediator): X -> M -> Y
draw(axes[0], [("X", "M"), ("M", "Y")],
     {"X": (0, 0), "M": (1, 0), "Y": (2, 0)},
     "Chain / mediator\nX → M → Y")

# Fork (confounder): X <- Z -> Y
draw(axes[1], [("Z", "X"), ("Z", "Y")],
     {"Z": (1, 1), "X": (0, 0), "Y": (2, 0)},
     "Fork / confounder\nX ← Z → Y")

# Collider: X -> C <- Y
draw(axes[2], [("X", "C"), ("Y", "C")],
     {"C": (1, 0), "X": (0, 1), "Y": (2, 1)},
     "Collider\nX → C ← Y")

plt.tight_layout()
plt.show()

* **Chain** $X \to M \to Y$: $M$ is a **mediator** (it transmits the effect).
* **Fork** $X \leftarrow Z \to Y$: $Z$ is a **confounder** (common cause).
* **Collider** $X \to C \leftarrow Y$: $C$ is a **common effect**.


Any path is a sequence of these: each interior node is a mediator, a confounder, or a
collider *on that path*.


## Open and blocked paths

A path carries association between its endpoints only if it is **open**. Two rules decide:

1. A **chain** or **fork** node is *open by default* and becomes **blocked** if you condition
   on it (adjust / stratify / put it in the regression).
2. A **collider** node is *blocked by default* and becomes **open** if you condition on it
   (or on any of its descendants).


A path is **blocked** if *at least one* of its interior nodes is blocked; otherwise it is
**open**. Two variables are (conditionally) independent when **every** path between them is
blocked.


Sanity check on the example DAG: $X \leftarrow A \to Y$ is open (fork, nothing conditioned) —
that is the ice-cream/temperature confounding of Lecture I. Condition on $A$ and it closes.


## The golden rule: adjust for forks, not colliders

The three structures behave *oppositely* when you condition (adjust/stratify) on the middle node:

| Structure | Middle node | Adjust? | Why |
|:--|:--|:--:|:--|
| Fork $X \leftarrow Z \to Y$ | confounder | **Yes** | blocks the backdoor path |
| Chain $X \to M \to Y$ | mediator | **No**\* | it *is* the effect you want |
| Collider $X \to C \leftarrow Y$ | collider | **No!** | *creates* a spurious association |

: {tbl-colwidths="[36,18,12,34]"}

\* Adjust for a mediator only if you specifically want the *direct* (not total) effect.

"Just throw every variable into the regression" is **wrong** — controlling for a collider
or a mediator *introduces* bias. Choosing controls is a causal decision, not a statistical
one [@cinelli2020crash].


## Confounders: the backdoor

A **backdoor path** is a non-causal path from $X$ to $Y$ that starts with an arrow *into*
$X$ (e.g. $X \leftarrow Z \to Y$). Backdoor paths are the graph-language for confounding.

**Backdoor criterion:** a set of variables $S$ suffices to identify the causal effect of
$X$ on $Y$ if $S$ **blocks every backdoor path** and contains **no descendants of $X$**
[@pearl2009causality].


When such an $S$ exists, Lecture I's **direct standardization** applies with $S$ in place
of the stratifying variable:
$$ P(Y \mid do(X{=}x)) = \sum_{s} P(Y \mid X{=}x, S{=}s)\, P(S{=}s). $$
The backdoor criterion is what tells you the formula is *legitimate*. It was legitimate
for kidney-stone size because stone size is a fork.


## The backdoor procedure, by hand

Given a DAG, a treatment $X$ and an outcome $Y$:

1. **List every path** from $X$ to $Y$ (ignore arrow direction while tracing; visit no node
   twice).
2. **Set aside the causal paths** — those where every arrow points away from $X$ toward $Y$.
   Never block these (that would be adjusting for a mediator).
3. **Classify the rest.** A path whose first arrow points *into* $X$ is a **backdoor path** —
   mark it open or blocked (forks/chains open, colliders closed). A path whose first arrow
   points *out of* $X$ but that is not causal — it runs into a collider, e.g.
   $X \to C \leftarrow Y$ — is **neither**: blocked by default, and it needs no action *as
   long as you leave $C$ and its descendants alone*.
4. **Choose $S$** that blocks every open backdoor path — a fork or chain node on each —
   containing **no collider or collider-descendant on *any* path** (backdoor or not) and
   **no descendant of $X$**.
5. **Report $S$** (there may be several valid choices; the smallest is usually best).


Then adjust for $S$ by whichever estimator you like — regression, stratification, matching,
propensity scores (next).


## Worked example

Back to the exercise DAG ($X$ = exercise, $Y$ = heart health):

$$ A \to X,\; A \to Y,\; M \to X,\; M \to D,\; D \to Y,\; X \to W,\; W \to Y $$

1. Paths from $X$ to $Y$: $X \to W \to Y$; $X \leftarrow A \to Y$; $X \leftarrow M \to D \to Y$.
2. Causal path: $X \to W \to Y$ — leave $W$ alone.
3. Backdoor paths: $X \leftarrow A \to Y$ (open, fork at $A$) and $X \leftarrow M \to D \to Y$
   (open, fork at $M$, chain at $D$). No colliders anywhere.
4. Block the first with $A$; block the second with $M$ *or* $D$.
5. Valid adjustment sets: $\{A, M\}$, $\{A, D\}$, or $\{A, M, D\}$. **Not** $\{A\}$ alone
   (leaves a backdoor open); **not** anything containing $W$ (a descendant of $X$).


In regression terms: `Y ~ X + A + M` recovers the total effect of exercise;
`Y ~ X + A + M + W` does not.


## Your turn: pick the adjustment set

Tutoring ($X$) and final grade ($Y$), with:
$\text{GPA}_{\text{prior}} \to X$, $\text{GPA}_{\text{prior}} \to Y$,
$X \to \text{Hours} \to Y$, and $X \to \text{HonorRoll} \leftarrow Y$.

Which of these adjustment sets are valid for the **total** effect of $X$ on $Y$?

* (a) $\{\text{GPA}_{\text{prior}}\}$
* (b) $\{\text{GPA}_{\text{prior}}, \text{Hours}\}$
* (c) $\{\text{GPA}_{\text{prior}}, \text{HonorRoll}\}$
* (d) $\{\}$ (nothing)


Pause and decide before you read on: run the five steps.


## Your turn: the answer

In [ ]:
#| echo: false
#| fig-align: center
#| label: tutoring-dag
from slideUtilities import OKABE_ITO

tutor_edges = [("GPA\nprior", "X"), ("GPA\nprior", "Y"), ("X", "Hours"), ("Hours", "Y"),
               ("X", "Honor\nroll"), ("Y", "Honor\nroll")]
tutor_pos = {"X": (0, 0), "GPA\nprior": (1, 1.25), "Hours": (1, 0.3),
             "Honor\nroll": (1, -0.95), "Y": (2, 0)}
roles = {"GPA\nprior": OKABE_ITO[2], "Hours": OKABE_ITO[0], "Honor\nroll": OKABE_ITO[3]}

fig, ax = plt.subplots(figsize=(4.6, 3.2))
draw(ax, tutor_edges, tutor_pos, node_color=roles, node_size=1700, font_size=9,
     title="X = tutoring, Y = final grade\n"
           "green = confounder, orange = mediator, red = collider")
ax.set_xlim(-0.3, 2.3); ax.set_ylim(-1.35, 1.6)
plt.tight_layout(); plt.show()

| Path | Type | Action |
|:--|:--|:--|
| $X \to \text{Hours} \to Y$ | causal (open) | leave alone |
| $X \leftarrow \text{GPA} \to Y$ | backdoor (open fork) | block with GPA |
| $X \to \text{Honor} \leftarrow Y$ | **neither** (blocked collider) | do not touch |

: {tbl-colwidths="[44,34,22]"}


**Only (a).** (b) blocks the mediator; (c) opens the collider; (d) leaves the backdoor
open. Restricting the study to honor-roll students — the "cleaner subset" — is (c) in
disguise.


## Colliders: adjusting can *create* bias

Conditioning on a collider (or its descendant) opens a spurious path. Classic example:
suppose **talent** and **looks** are independent in the general population, but both help an
actor get **famous**.

In [ ]:
#| echo: true
import numpy as np
rng = np.random.default_rng(701)
n = 10_000

talent = rng.normal(0, 1, n)
looks  = rng.normal(0, 1, n)          # independent of talent by construction
famous = (talent + looks + rng.normal(0, 0.5, n)) > 1.5   # collider: common effect

corr_all    = np.corrcoef(talent, looks)[0, 1]
corr_famous = np.corrcoef(talent[famous], looks[famous])[0, 1]
print(f"corr(talent, looks) overall        = {corr_all:+.2f}")
print(f"corr(talent, looks) among famous    = {corr_famous:+.2f}")

In [ ]:
#| echo: false
#| fig-align: center
#| label: collider-scatter
fig, ax = plt.subplots(figsize=(5.6, 3.9))
ax.scatter(talent[~famous], looks[~famous], s=4, color="#bbb", label="not famous")
ax.scatter(talent[famous], looks[famous], s=6, color=OKABE_ITO[0], label="famous")
grid = np.linspace(-4, 4, 2)
for mask, color, name in [(np.ones(n, bool), "#555", "everyone"), (famous, OKABE_ITO[3], "famous only")]:
    slope, icpt = np.polyfit(talent[mask], looks[mask], 1)
    ax.plot(grid, icpt + slope * grid, color=color, lw=2,
            label=f"OLS line, {name} (r = {np.corrcoef(talent[mask], looks[mask])[0, 1]:+.2f})")
ax.plot(grid, 1.5 - grid, ls="--", color="black", lw=1, label="talent + looks = 1.5")
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel("talent"); ax.set_ylabel("looks")
ax.legend(fontsize=7, loc="lower left", markerscale=2)
plt.tight_layout(); plt.show()

Among the famous, talent and looks are **negatively** correlated — "why are talented
celebrities so often unattractive?" — even though they are unrelated in general. We
manufactured the correlation by conditioning on fame.


## Collider bias, and a collider's descendants

This is **collider bias**. Epidemiologists call it *selection bias* or *Berkson's paradox*,
because restricting a sample is a way of conditioning. It is **not** the selection-bias term
of Lecture I's decomposition (a baseline difference between treated and untreated): that
one is a fork, this one is a collider.

Conditioning on a **descendant** of the collider works too. Follower count is caused by
fame; select on it instead of fame itself:

In [ ]:
#| echo: true
followers = 3 * famous + rng.normal(0, 1, n)       # a noisy proxy for fame
sel = followers > 2
print(f"corr(talent, looks | followers > 2) = {np.corrcoef(talent[sel], looks[sel])[0, 1]:+.2f}"
      f"   ({famous[sel].mean():.0%} of the selected are famous)")

A noisy proxy for a collider is still a collider: it opens the path *partially*.


## Mediators: don't "control away" your effect

If a job-training program raises earnings *by* teaching skills:

$$ \text{Training} \rightarrow \text{Skills} \rightarrow \text{Earnings} $$

Controlling for **skills** would block the very pathway the program works through, making the
program look useless. This is **overcontrol bias**.


Rule of thumb for a **total** effect: adjust for common *causes*, never for anything on the
causal path or for common *effects* [@cinelli2020crash].


## Estimating effects under "no unmeasured confounders"

Suppose we've drawn the DAG and found an adjustment set $S$ that satisfies the backdoor
criterion. In Lecture I's language this is **conditional ignorability**:
$$ \{Y(0), Y(1)\} \perp\!\!\!\perp T \mid S $$
— randomization's independence, now holding only *within* levels of $S$. The DAG is how we
argue it; it cannot be checked from data.

From here on the treatment is $T$, as in Lecture I; $X$ was convenient while tracing paths.


## Four ways to adjust for $S$

Once $S$ is chosen, any of these recovers the effect:

1. **Regression adjustment** — include $S$ as covariates; read off the coefficient on $T$
   (this assumes the linear form is right: the regression is an estimator, not a licence).
2. **Stratification / standardization** — estimate the effect within levels of $S$, then
   average with the **population's** stratum shares (→ ATE); use the *treated* group's
   shares and you get the ATT.
3. **Matching** — pair each treated unit with similar control unit(s) on $S$.
4. **Propensity scores** — match or weight on $e(S) = P(T{=}1 \mid S)$, a one-number summary.


## Regression adjustment on a simulation

A confounder $Z$ (how sick a patient is) makes treatment more likely **and** makes the
outcome worse. The true effect of $T$ is $3$.

In [ ]:
#| echo: true
import numpy as np, pandas as pd, statsmodels.formula.api as smf

rng = np.random.default_rng(0)
n = 4000
Z = rng.normal(0, 1, n)                                         # sickness
T = (rng.uniform(size=n) < 1 / (1 + np.exp(-Z))).astype(int)   # sicker -> more likely treated
Y = 3.0 * T - 2.0 * Z + rng.normal(0, 1, n)                    # sicker -> worse; TRUE effect = 3.0
df = pd.DataFrame({"Y": Y, "T": T, "Z": Z})

naive = smf.ols("Y ~ T", data=df).fit().params["T"]
adj   = smf.ols("Y ~ T + Z", data=df).fit().params["T"]
print(f"True effect        = 3.00")
print(f"Naive (ignore Z)   = {naive:.2f}   <- biased")
print(f"Adjusted (+ Z)     = {adj:.2f}   <- recovers truth")

Predict the sign before you run it: $Z$ raises treatment *and* lowers $Y$, so the naive
estimate is biased **down**. The treatment looks less than half as effective as it is,
because the sickest were the ones treated.


## Stratification: Lecture I's move on a continuous $Z$

Cut $Z$ into strata, compare treated and control *within* each, then average with the
population's stratum shares — direct standardization with $S$ in place of stone size.

In [ ]:
#| echo: true
def stratified(df, k):
    """Direct standardization over k equal-size strata of Z."""
    q = pd.qcut(df["Z"], k, labels=False)
    means = df.groupby([q, "T"])["Y"].mean().unstack()     # rows: strata, cols: T = 0, 1
    within = means[1] - means[0]                           # effect inside each stratum
    return within, (within * q.value_counts(normalize=True).sort_index()).sum()

for k in (5, 10, 20):
    print(f"{k:2d} strata: standardized effect = {stratified(df, k)[1]:.2f}")

In [ ]:
#| echo: false
#| fig-align: center
#| label: strata
within5, std5 = stratified(df, 5)
std10, std20 = stratified(df, 10)[1], stratified(df, 20)[1]

fig, ax = plt.subplots(figsize=(6.4, 2.5))
labels = ["naive"] + [f"Q{i + 1}" for i in range(5)]
ax.bar(labels, [naive, *within5], color=["#999"] + [OKABE_ITO[5]] * 5)
for i, v in enumerate([naive, *within5]):
    ax.text(i, v - 0.1, f"{v:.2f}", ha="center", va="top", fontsize=9, color="white")
ax.axhline(3.0, ls="--", color="black", lw=1, label="true effect = 3.0")
ax.set_ylabel("treated − control")
ax.set_xlabel("quintile of Z (Q1 = healthiest)")
ax.set_ylim(0, 3.8)
ax.set_title(f"Standardized: 5 strata {std5:.2f}, 10 strata {std10:.2f}, "
             f"20 strata {std20:.2f}", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

Coarse strata leave **residual confounding** inside each stratum; finer strata close the
gap. Regression is the limit of infinitely fine strata — bought with a linearity assumption.


## Propensity scores in one slide

When $S$ is high-dimensional, matching directly is hard. Rosenbaum and Rubin showed it is
enough to match or weight on the scalar **propensity score** $e(S) = P(T{=}1\mid S)$
[@rosenbaum1983central].

In [ ]:
#| echo: true
from sklearn.linear_model import LogisticRegression

# Estimate propensity, then inverse-propensity weighting (IPW) for the ATE
ps = LogisticRegression().fit(df[["Z"]], df["T"]).predict_proba(df[["Z"]])[:, 1]
w = np.where(df["T"] == 1, 1 / ps, 1 / (1 - ps))
ate_ipw = (np.sum(w * df["T"] * df["Y"]) / np.sum(w * df["T"])
           - np.sum(w * (1 - df["T"]) * df["Y"]) / np.sum(w * (1 - df["T"])))
print(f"IPW estimate of ATE = {ate_ipw:.2f}   (true = 3.0)")

Inverse-propensity weighting builds a "pseudo-population" in which treatment is unrelated to
$S$ — mimicking a randomized experiment from observational data.


## Positivity: can we see it?

In [ ]:
#| echo: false
#| fig-align: center
#| label: overlap
fig, ax = plt.subplots(figsize=(6.4, 3.0))
bins = np.linspace(0, 1, 31)
ax.hist(ps[df["T"] == 1], bins=bins, alpha=0.5, color=OKABE_ITO[0], label="treated")
ax.hist(ps[df["T"] == 0], bins=bins, alpha=0.5, color=OKABE_ITO[2], label="control")
ax.set_xlabel("estimated propensity $e(Z)$"); ax.set_ylabel("patients")
ax.set_title("Overlap: both groups span nearly the same propensity range", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"treated: e from {ps[df['T'] == 1].min():.2f} to {ps[df['T'] == 1].max():.2f};  "
      f"control: {ps[df['T'] == 0].min():.2f} to {ps[df['T'] == 0].max():.2f};  "
      f"largest weight = {w.max():.1f}")

If one histogram had mass where the other has none, some units would have no comparable
counterpart. An $e$ of $0.01$ gives a weight of $100$: a tight standard error around a number
that means nothing.


## Same adjustment set, same answer

In [ ]:
#| echo: false
#| fig-align: center
#| label: estimators
fig, ax = plt.subplots(figsize=(6.4, 3.2))
names = ["naive\nY ~ T", "regression\nY ~ T + Z", "stratified\n20 strata of Z", "IPW on\ne(Z)"]
vals = [naive, adj, std20, ate_ipw]
ax.bar(names, vals, width=0.55, color=[OKABE_ITO[3]] + [OKABE_ITO[2]] * 3)
for i, v in enumerate(vals):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", va="bottom")
ax.axhline(3.0, ls="--", color="black", label="true effect = 3.0")
ax.set_ylabel("estimated effect of T"); ax.set_ylim(0, 3.8)
ax.set_title("Same data, same adjustment set — three estimators", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

Identification (which $S$) and estimation (which estimator) are separate decisions. A
*wrong* $S$ gives a wrong answer with a tight standard error — no estimator rescues a bad graph.


## When confounders are *unobserved*: natural experiments

Adjustment only works for confounders we can **measure**. When we can't, we look for
"nature's randomization" — sources of variation in treatment that are *as good as random*
[@angrist2014mastering].

Three workhorse designs:

* **Difference-in-Differences (DiD)**,
* **Instrumental Variables (IV)**,
* **Regression Discontinuity (RD)**.


## Difference-in-Differences

Compare the **change** in a treated group to the **change** in a control group over time.
This differences out any *fixed* differences between the groups.

In [ ]:
#| echo: false
#| fig-align: center
#| label: did
t = [-1, 0, 1]
treated_obs = [17, 20, 26]      # treated group: pre-period, before, after the policy
control     = [21, 24, 27]      # control group
counterfact = [20, 20 + (control[2] - control[1])]   # treated had it followed control's trend

fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(t, treated_obs, "o-", label="Treated (observed)", color=OKABE_ITO[3])
ax.plot(t, control, "o-", label="Control", color=OKABE_ITO[2])
ax.plot([0, 1], counterfact, "o--", label="Treated (counterfactual)", color="#999")
ax.axvline(0.5, color="#ccc", lw=1)
ax.text(0.52, 16.6, "policy", color="#777", fontsize=8)
ax.annotate("", xy=(1, treated_obs[2]), xytext=(1, counterfact[1]),
            arrowprops=dict(arrowstyle="<->", color="black"))
ax.text(1.04, (treated_obs[2] + counterfact[1]) / 2, "DiD\n= 3", va="center")
ax.set_xticks(t); ax.set_xticklabels(["Pre", "Before", "After"])
ax.set_xlim(-1.3, 1.45)
ax.set_ylabel("Outcome"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

|  | Before | After | Change |
|:--|--:|--:|--:|
| Treated | 20 | 26 | +6 |
| Control | 24 | 27 | +3 |

DiD $= (26 - 20) - (27 - 24) = \mathbf{3}$.

Key assumption: **parallel trends** — absent treatment, both groups would have moved
together. The only check you can run is on the *pre* period: here both rose by 3.


Card and Krueger used DiD to show that a New Jersey minimum-wage increase did **not** reduce
fast-food employment relative to neighboring Pennsylvania [@card1994minimum].


## Instrumental variables & regression discontinuity

**Instrumental variable (IV)**

An **instrument** $L$ shifts the treatment and affects the outcome *only through* it:
$$ L \rightarrow T \rightarrow Y, \quad L \not\rightarrow Y \text{ directly}. $$

- Vietnam **draft lottery** as an instrument for military service.
- **Distance to college** as an instrument for years of schooling.

$L$ acts like the coin flip we wish we had run.

**Regression discontinuity (RD)**

Treatment switches on at a **cutoff** of a running variable. Units just above vs. just below
are comparable.

- Scholarship awarded at a **test-score threshold**.
- Class-size caps that trigger a new section.

Compare outcomes in a narrow window around the cutoff.


## An instrument, in graph terms

In [ ]:
#| echo: false
#| fig-align: center
#| label: iv-dag
iv_pos = {"L\nlottery": (0, 0), "T": (1, 0), "Y": (2, 0), "U\nunobserved": (1.5, 1)}
fig, ax = plt.subplots(figsize=(6.4, 2.8))
draw(ax, [("L\nlottery", "T"), ("T", "Y")], iv_pos, node_size=2000, font_size=9)
U = nx.DiGraph([("U\nunobserved", "T"), ("U\nunobserved", "Y")])
nx.draw_networkx_nodes(U, iv_pos, ax=ax, nodelist=["U\nunobserved"], node_color="white",
                       node_size=2900, edgecolors="#999", linewidths=1.2)
nx.draw_networkx_labels(U, iv_pos, ax=ax, labels={"U\nunobserved": "U\nunobserved"},
                        font_size=9, font_color="#777")
nx.draw_networkx_edges(U, iv_pos, ax=ax, node_size=2900, style="dashed",
                       edge_color="#999", arrowsize=22)
nx.draw_networkx_edges(nx.DiGraph([("L\nlottery", "Y")]), iv_pos, ax=ax, node_size=2000,
                       style="dashed", edge_color="#c0392b", width=2, arrowsize=22,
                       connectionstyle="arc3,rad=0.3")
ax.text(1, -0.55, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
        bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
ax.text(1, -0.8, "no direct L → Y (exclusion)", color="#c0392b", fontsize=9,
        ha="center", va="top")
ax.set_xlim(-0.4, 2.4); ax.set_ylim(-1.1, 1.35)
plt.tight_layout(); plt.show()

1. **Relevance** — the arrow $L \to T$ exists (the lottery really changes who serves).
2. **Exclusion** — no arrow $L \to Y$: the lottery affects earnings *only through* service.
3. **Independence** — no backdoor path from $L$ to $Y$: nothing points into $L$ (it is random).


The unobserved $U$ confounds $T$ and $Y$, so adjustment is impossible — but $L$ is untouched
by $U$, and that is all IV needs.


## Three designs, three assumptions

| Design | As-good-as-random variation | Assumption | How it fails |
|:--|:--|:--|:--|
| **DiD** | the *change* in treated vs. control | **parallel trends** | pre-period trends already diverge |
| **IV** | an instrument $L$ | **relevance** ($L \to T$), **exclusion** (no $L \to Y$), **independence** (no backdoor $L$–$Y$) | the lottery shifts earnings via draft *avoidance* (staying in college) |
| **RD** | a cutoff on a running variable | **nothing else jumps at the cutoff**; no precise manipulation | people bunch just above the threshold |

: {tbl-colwidths="[10,24,38,28]"}

Each design swaps "no unmeasured confounders" for a *different*, often more credible,
assumption that data cannot fully verify. No free lunch.


## Assumptions you can't skip

Every observational causal estimate relies on assumptions [@hernan2020whatif]:

* **Conditional ignorability** — $\{Y(0),Y(1)\} \perp\!\!\!\perp T \mid S$: the adjustment set
  blocks every backdoor path. The DAG argues it; data cannot check it.
* **Positivity (overlap)** — every unit has a nonzero chance of each treatment ($0 < e(S) < 1$).
* **SUTVA** — the assumption Lecture I's switching equation
  $Y^{\text{obs}} = T\,Y(1) + (1-T)\,Y(0)$ quietly made: one unit's treatment doesn't affect
  another's outcome, and there is only one version of "treated."
* **Correct graph** — the DAG reflects reality.


Since ignorability is untestable, good practice includes a **sensitivity analysis**:
how strong would an unmeasured confounder have to be to overturn the conclusion?


## Tools and the modern workflow

* **[DoWhy](https://www.pywhy.org/dowhy/)** — model (DAG) → identify → estimate → refute; a
  clean four-step API [@dowhy2020].
* **[EconML](https://econml.azurewebsites.net/)** — ML-based heterogeneous treatment effects (CATE).
* **[dagitty](https://dagitty.net/)** — draw a DAG in the browser; it tells you the valid
  adjustment sets.
* Classical estimators live in **statsmodels**, **scikit-learn**, and **linearmodels**.


The workflow is always: **draw the graph → identify an estimand → estimate → challenge it
with robustness checks.** The graph comes *first*.


## Summary

1. **A causal DAG encodes your assumptions, not the data's.** Nodes are variables, arrows are
   direct causes, and every path is built from chains, forks, and colliders.
2. **Chains and forks are open until you condition on them; colliders are closed until you
   do** (or condition on a descendant). A path transmits association only if it is open.
3. **The backdoor procedure sorts every path into causal, backdoor or neither**, then blocks
   every open backdoor with non-descendants of $X$ — never a mediator, never a collider.
4. **Identification first, estimation second**: given a valid $S$, regression,
   stratification and IPW agree; with a wrong $S$ none of them can help.
5. **When confounders are unmeasured, use natural experiments** — DiD (parallel trends), IV
   (relevance, exclusion, independence), RD (nothing else jumps at the cutoff) — and state
   the standing assumptions (ignorability, positivity, SUTVA).


## Practice scenarios (self-check)

For each scenario, draw the DAG, name the **confounder**, the tempting-but-wrong control
(a mediator or collider), and a **design** you would use. Then open the intended answers.

**A. Product / tech.** Does adding a **"free shipping" badge** on product pages increase
**purchases**? You have historical logs; badges were shown more often on popular items.

**B. Health.** Does a new **fitness-tracking app** reduce users' **resting heart rate**? People
who download it are already more health-conscious.

**C. Education.** Does an **optional tutoring program** raise **final grades**? Students
self-select into tutoring, and struggling students are more likely to sign up.

**D. Policy.** Does a city's **new bike-lane network** reduce **traffic accidents**? It was built
in denser, wealthier neighborhoods first.

**Intended answers**

- **A.** Confounder: item **popularity** (drives both the badge and purchases). Tempting
  control: **"add-to-cart"** — a mediator on badge → add-to-cart → purchase. Design: an A/B
  test that randomizes the badge, or adjust for popularity with a propensity score.
- **B.** Confounder: **health-consciousness**. Tempting controls: **steps per day** (a
  mediator — the app works by getting people moving) and **"posted an app review"** (a
  collider — caused by both installing the app and by feeling the benefit). Design:
  randomize a free premium tier as an encouragement (an instrument), or DiD on users' heart
  rate before and after installing.
- **C.** This is the lecture's tutoring DAG. Confounder: **prior GPA** (struggling students
  sign up *and* score lower, so the naive estimate is biased down). Tempting controls:
  **study hours** (mediator) and **honor roll** (collider). Design: adjust for prior GPA; or
  RD if tutoring is assigned below a GPA cutoff.
- **D.** Confounder: neighborhood **density / wealth**. Tempting control: **cyclist volume** —
  a mediator (lanes → more cyclists → accidents). Design: DiD comparing accident trends in
  neighborhoods that got lanes early vs. late, checking pre-period trends.



## References

- [*Causal Inference: What If*](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) — Hernán & Robins (free PDF)
- [*Causal Inference: A Primer*](http://bayes.cs.ucla.edu/PRIMER/) — Pearl, Glymour & Jewell
- ["A Crash Course in Good and Bad Controls"](https://ftp.cs.ucla.edu/pub/stat_ser/r493.pdf) — Cinelli, Forney & Pearl
- [DoWhy documentation](https://www.pywhy.org/dowhy/) and [dagitty.net](https://dagitty.net/)

## Bibliography
